# Prepare dataset

In [ ]:
# imports

import os
import torch
import pandas as pd
from torch.utils.data.dataset import Dataset
from transformers import AutoTokenizer

In [ ]:
articles_path = 'D:\\univ_subj\\M_An_I\\Sem 2\\BioMedical NLP\\GutBrainIE\\GutBrainIE_Full_Collection_2026\\Articles\\csv_format'
test_articles_path = 'D:\\univ_subj\\M_An_I\\Sem 2\\BioMedical NLP\\GutBrainIE\\Task_gutbrain_Testset1raw\\articles_test.csv'
annotations_path = 'D:\\univ_subj\\M_An_I\\Sem 2\\BioMedical NLP\\GutBrainIE\\GutBrainIE_Full_Collection_2026\\Annotations'

In [ ]:
categories_to_ids = {
    'anatomical location': 0,
    'animal': 1,
    'biomedical technique': 2,
    'bacteria': 3,
    'chemical': 4,
    'dietary supplement': 5,
    'DDF': 6,
    'drug': 7,
    'food': 8,
    'gene': 9,
    'human': 10,
    'microbiome': 11,
    'statistical technique': 12,
    'none': 13
}

ids_to_categories = {v: k for k, v in categories_to_ids.items()}

In [ ]:
def get_annotations(annotations_file):
    annotations_df = pd.read_csv(annotations_file, sep='|')
    annotations_df = annotations_df.drop(columns=['annotator'])
    annotations_df['location'] = annotations_df['location'].map(lambda x: 1 if x == 'title' else 0)
    annotations_df['label'] = annotations_df['label'].map(lambda x: categories_to_ids[x])
    print(annotations_df.head())
    return annotations_df

def get_articles(articles_file, tokenizer):
    old_articles_df = pd.read_csv(articles_file, sep='|')
    articles_df = old_articles_df[['pmid']].copy()
    old_articles_df['title'] = old_articles_df['title'].map(lambda x: x + '. ')
    texts = old_articles_df['title'] + '. ' + old_articles_df['abstract']
    articles_df['text'] = texts
    articles_df['added_chars_no'] = old_articles_df['title'].map(lambda x: len(x) + 2)
    texts = texts.map(lambda x: tokenizer(x, truncation=True, padding=True, return_offsets_mapping=True))
    articles_df['input_ids'] = texts.map(lambda x: x['input_ids'])
    articles_df['offset_mapping'] = texts.map(lambda x: x['offset_mapping'])
    print(articles_df.head())
    return articles_df


In [ ]:
class GBDataset(Dataset):
    def __init__(self, articles_df, annotations_df=None):
        self.annotations_df = annotations_df
        self.articles_df = articles_df

        self.dataset = []

        mismatches = 0

        for idx in range(self.articles_df.shape[0]):
            start_labels = torch.zeros(len(self.articles_df["input_ids"][idx]), dtype=torch.uint8)
            end_labels = torch.zeros(len(self.articles_df["input_ids"][idx]), dtype=torch.uint8)
            span_idxs = []
            span_labels = []
            start_token_to_orig = {}
            end_token_to_orig = {}
            pmid = self.articles_df["pmid"][idx]
            if self.annotations_df is not None:
                for row in self.annotations_df[self.annotations_df['pmid'] == pmid].itertuples(name=None):
                    start_token = None
                    end_token = None

                    ann_start_idx = row[2]
                    ann_end_idx = row[3]

                    # if the entity is in the abstract, shift the idx with the number of chars
                    # added at the start of the string for the title
                    if row[4] == 0:
                        ann_start_idx += self.articles_df['added_chars_no'][idx]
                        ann_end_idx += self.articles_df['added_chars_no'][idx]

                    for i, (offset_start, offset_end) in enumerate(self.articles_df["offset_mapping"][idx]):
                        # Skip special tokens like [CLS] or [SEP] which have (0,0) offsets
                        if offset_start == 0 and offset_end == 0:
                            continue


                        # 1. Capture the start: First token that contains char_start
                        if start_token is None and offset_start <= ann_start_idx < offset_end:
                            start_token = i

                        # 2. Capture the end: Token that contains the last character
                        if offset_start <= ann_end_idx < offset_end:
                            end_token = i


                    if start_token is not None and end_token is not None:
                        start_labels[start_token] = 1
                        end_labels[end_token] = 1
                        span_idxs.append([start_token, end_token])
                        span_labels.append(row[6])
                        start_token_to_orig[start_token] = ann_start_idx if row[4] == 1 else ann_start_idx - self.articles_df['added_chars_no'][idx]
                        end_token_to_orig[end_token] = ann_end_idx if row[4] == 1 else ann_end_idx - self.articles_df['added_chars_no'][idx]

                        # 1. Get the original text for comparison
                        original_entity = self.articles_df['text'][idx][ann_start_idx:ann_end_idx + 1]

                        # 2. Get the tokens from the input_ids
                        # input_ids is the tensor/list from the tokenizer encoding
                        token_ids = self.articles_df['input_ids'][idx][start_token : end_token + 1]
                        decoded_entity = tokenizer.decode(token_ids).strip()


                        # 3. Print and compare
                        if original_entity.lower() != decoded_entity.lower():
                            print(f"Mismatch found!")
                            print(f"Original: '{original_entity}'")
                            print(f"Decoded:  '{decoded_entity}'")
                            print(f"Indices:  {start_token}:{end_token}")
                            mismatches += 1

            self.dataset.append({
                "pmid":            pmid,
                "location":        self.annotations_df[self.annotations_df['pmid'] == pmid]['location'] if self.annotations_df is not None else None,
                "input_ids":       torch.tensor(self.articles_df.iloc[idx]['input_ids'], dtype=torch.long),
                "start_labels":    start_labels,
                "end_labels":      end_labels,
                "span_idxs":       span_idxs,
                "span_labels":     span_labels,
                "start_token_to_orig": start_token_to_orig,
                "end_token_to_orig": end_token_to_orig,
                "abstract_start_idx": self.articles_df['added_chars_no'][idx],
                "offset_mapping": self.articles_df['offset_mapping'][idx],
            })

    def __len__(self):
        return self.articles_df.shape[0]

    def __getitem__(self, idx):
        return self.dataset[idx]


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("thomas-sounack/BioClinical-ModernBERT-base")

In [ ]:
bronze_annotations = get_annotations(os.path.join(annotations_path, 'Train/bronze_quality/csv_format/train_bronze_entities.csv'))
bronze_articles = get_articles(os.path.join(articles_path, 'articles_train_bronze.csv'), tokenizer)
bronze_dataset = GBDataset(bronze_articles, bronze_annotations)
torch.save(bronze_dataset.dataset, os.path.join('./datasets', 'bronze_dataset.pt'))

In [ ]:
silver_annotations = get_annotations(os.path.join(annotations_path, 'Train/silver_quality/csv_format/train_silver_entities.csv'))
silver_articles = get_articles(os.path.join(articles_path, 'articles_train_silver.csv'), tokenizer)
silver_dataset = GBDataset(silver_articles, silver_annotations)
torch.save(silver_dataset.dataset, os.path.join('./datasets', 'silver_dataset.pt'))

In [ ]:
silver2025_annotations = get_annotations(os.path.join(annotations_path, 'Train/silver_quality/csv_format/train_silver_2025_entities.csv'))
silver2025_articles = get_articles(os.path.join(articles_path, 'articles_train_silver_2025.csv'), tokenizer)
silver2025_dataset = GBDataset(silver2025_articles, silver2025_annotations)
torch.save(silver2025_dataset.dataset, os.path.join('./datasets', 'silver2025_dataset.pt'))

In [ ]:
gold_annotations = get_annotations(os.path.join(annotations_path, 'Train/gold_quality/csv_format/train_gold_entities.csv'))
gold_articles = get_articles(os.path.join(articles_path, 'articles_train_gold.csv'), tokenizer)
gold_dataset = GBDataset(gold_articles, gold_annotations)
torch.save(gold_dataset.dataset, os.path.join('./datasets', 'gold_dataset.pt'))

In [ ]:
dev_annotations = get_annotations(os.path.join(annotations_path, 'Dev/csv_format/dev_entities.csv'))
dev_articles = get_articles(os.path.join(articles_path, 'articles_dev.csv'), tokenizer)
dev_dataset = GBDataset(dev_articles, dev_annotations)
torch.save(dev_dataset.dataset, os.path.join('./datasets', 'dev_dataset.pt'))

In [ ]:
test_articles = get_articles(test_articles_path, tokenizer)
test_dataset = GBDataset(test_articles)
torch.save(test_dataset.dataset, os.path.join('./datasets', 'test_dataset.pt'))

## References

* Sounack, Thomas, et al. "Bioclinical modernbert: A state-of-the-art long-context encoder for biomedical and clinical nlp." arXiv preprint arXiv:2506.10896 (2025).
* Li, Xiaoya, et al. "A unified MRC framework for named entity recognition." Proceedings of the 58th annual meeting of the association for computational linguistics. 2020.
* Gemini, Claude, Chat GPT